In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/technical_gdelt_merged_2020_2025.csv
/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/__results__.html
/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/technical_gdelt_market_merged_2020_2025.csv
/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/__output__.json
/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/custom.css


In [2]:
import numpy as np
import pandas as pd

# Target Construction and Model-Ready Dataset

This notebook creates the prediction target for the stock-movement study.

The target is based on the five-trading-day forward return of each stock.
Each observation at trading day `t` is linked to the stock price five
trading days later.

The forward return is then converted into three movement classes:

- `Up`
- `Neutral`
- `Down`

The notebook also prepares leakage-safe feature groups for later ablation,
stock-specific, pooled, and cross-stock generalisation experiments.

In [3]:
merged_file=("/kaggle/input/notebooks/phyothaw/05-data-merging-alignment-ipynb/technical_gdelt_market_merged_2020_2025.csv")
df = pd.read_csv(merged_file)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Shape: (7535, 37)
Columns:
['ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'SMA_50', 'SMA_200', 'EMA_50', 'EMA_200', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'ATR', 'ADX', 'date', 'gdelt_company_tone', 'gdelt_tone_missing', 'gdelt_tone_lag_1', 'gdelt_tone_mean_3', 'gdelt_tone_mean_5', 'gdelt_tone_std_5', 'gdelt_tone_change_1d', 'gdelt_abnormal_tone_20', 'market_close', 'market_return', 'vix_close', 'vix_change', 'market_return_lag_1', 'vix_close_lag_1', 'vix_change_lag_1']


,ticker,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,SMA_200,...,gdelt_tone_std_5,gdelt_tone_change_1d,gdelt_abnormal_tone_20,market_close,market_return,vix_close,vix_change,market_return_lag_1,vix_close_lag_1,vix_change_lag_1
0,NVDA,5.934968,5.963804,5.884506,5.963804,237536000,0.0,0.0,5.355496,4.491415,...,NaN,NaN,NaN,3257.850098,NaN,12.47,NaN,NaN,NaN,NaN
1,NVDA,5.844236,5.912100,5.819378,5.868349,205384000,0.0,0.0,5.375684,4.499142,...,0.862034,1.2191,NaN,3234.850098,-0.007060,14.02,0.124298,NaN,12.47,NaN
2,NVDA,5.775127,5.898177,5.749026,5.892956,262636000,0.0,0.0,5.396621,4.505810,...,0.460595,0.1011,NaN,3246.280029,0.003533,13.85,-0.012126,-0.007060,14.02,0.124298
3,NVDA,5.921294,6.010039,5.876301,5.964300,314856000,0.0,0.0,5.418107,4.513633,...,0.367547,-0.4970,NaN,3237.179932,-0.002803,13.79,-0.004332,0.003533,13.85,-0.012126
4,NVDA,5.960074,6.016751,5.920052,5.975486,277108000,0.0,0.0,5.436000,4.521973,...,0.217195,0.1376,NaN,3253.050049,0.004902,13.45,-0.024656,-0.002803,13.79,-0.004332


In [4]:
# Convert the date column and sort each stock chronologically.
#
# Correct chronological ordering is essential because the future-price
# target is created within each ticker using trading-day shifts.

df["date"] = pd.to_datetime(df["date"],errors="coerce")
df["ticker"] = (df["ticker"].astype(str).str.strip().str.upper())
df = (df.sort_values(["ticker", "date"]).reset_index(drop=True))

print("Invalid dates:", df["date"].isna().sum())
print("Duplicate ticker-date rows:",df.duplicated(subset=["ticker", "date"]).sum())
print("\nRows per ticker:")
print(df["ticker"].value_counts().sort_index())

Invalid dates: 0
Duplicate ticker-date rows: 0

Rows per ticker:
ticker
BRK-B    1507
CVX      1507
GE       1507
MSFT     1507
NVDA     1507
Name: count, dtype: int64


In [5]:
print(df[["ticker", "date", "Close", "Dividends", "Stock Splits"]].head(20))

   ticker       date       Close  Dividends  Stock Splits
0   BRK-B 2020-01-02  228.389999        0.0           0.0
1   BRK-B 2020-01-03  226.179993        0.0           0.0
2   BRK-B 2020-01-06  226.990005        0.0           0.0
3   BRK-B 2020-01-07  225.919998        0.0           0.0
4   BRK-B 2020-01-08  225.990005        0.0           0.0
5   BRK-B 2020-01-09  228.649994        0.0           0.0
6   BRK-B 2020-01-10  226.619995        0.0           0.0
7   BRK-B 2020-01-13  228.449997        0.0           0.0
8   BRK-B 2020-01-14  227.169998        0.0           0.0
9   BRK-B 2020-01-15  228.350006        0.0           0.0
10  BRK-B 2020-01-16  229.729996        0.0           0.0
11  BRK-B 2020-01-17  230.199997        0.0           0.0
12  BRK-B 2020-01-21  228.630005        0.0           0.0
13  BRK-B 2020-01-22  229.539993        0.0           0.0
14  BRK-B 2020-01-23  229.380005        0.0           0.0
15  BRK-B 2020-01-24  226.860001        0.0           0.0
16  BRK-B 2020

In [6]:
split_events = df[df["Stock Splits"] != 0][["ticker", "date", "Close", "Stock Splits"]]

print("Stock-split rows:", len(split_events))
display(split_events)

Stock-split rows: 5


,ticker,date,Close,Stock Splits
3412,GE,2021-08-02,61.193932,0.125
3771,GE,2023-01-04,55.027592,1.281
4082,GE,2024-04-02,134.443588,1.253
6417,NVDA,2021-07-20,18.547840,4.000
7144,NVDA,2024-06-10,121.579613,10.000


## Stock-Split Validation

Stock-split events are checked to confirm that the `Close` price series is properly adjusted.

Without adjustment, a split could create an artificial extreme return and produce incorrect technical indicators, forward returns, and target labels.

The split column is therefore used for data validation, but it will not be included as a main model feature.

In [7]:
# Inspect prices and returns around stock-split events.
# An adjusted price series should not show an artificial extreme return
# caused only by the split.

df["daily_return_check"] = (df.groupby("ticker")["Close"].pct_change())

split_indices = df.index[df["Stock Splits"] != 0]

rows_to_show = []

for idx in split_indices:
    start_idx = max(0, idx - 2)
    end_idx = min(len(df), idx + 3)

    rows_to_show.extend(
        range(start_idx, end_idx))

split_check = df.loc[
    sorted(set(rows_to_show)),
    ["ticker", "date", "Close", "Stock Splits", "daily_return_check"]]

display(split_check)

,ticker,date,Close,Stock Splits,daily_return_check
3410,GE,2021-07-29,64.673355,0.000,0.012186
3411,GE,2021-07-30,63.018795,0.000,-0.025583
3412,GE,2021-08-02,61.193932,0.125,-0.028957
3413,GE,2021-08-03,62.690331,0.000,0.024453
3414,GE,2021-08-04,62.599091,0.000,-0.001455
3769,GE,2022-12-30,51.272736,0.000,0.000478
3770,GE,2023-01-03,52.000919,0.000,0.014202
3771,GE,2023-01-04,55.027592,1.281,0.058204
3772,GE,2023-01-05,55.882011,0.000,0.015527
3773,GE,2023-01-06,56.391529,0.000,0.009118


## Five-Trading-Day Forward Return

The prediction target is based on the stock price five trading days after
each observation.

The calculation is performed separately for each ticker to prevent values
from one stock being shifted into another stock's rows.

In [8]:
# Select the closing price five trading rows ahead for each ticker.
#
# shift(-5) refers to five future trading observations, not five
# calendar days.

df["future_close_5d"] = (df.groupby("ticker")["Close"].shift(-5))

# Calculate the percentage return from the current closing price
# to the closing price five trading days later.

df["forward_return_5d"] = (df["future_close_5d"]/ df["Close"]- 1)
display(df[["ticker", "date", "Close", "future_close_5d", "forward_return_5d"]].head(10))

,ticker,date,Close,future_close_5d,forward_return_5d
0,BRK-B,2020-01-02,228.389999,228.649994,0.001138
1,BRK-B,2020-01-03,226.179993,226.619995,0.001945
2,BRK-B,2020-01-06,226.990005,228.449997,0.006432
3,BRK-B,2020-01-07,225.919998,227.169998,0.005533
4,BRK-B,2020-01-08,225.990005,228.350006,0.010443
5,BRK-B,2020-01-09,228.649994,229.729996,0.004723
6,BRK-B,2020-01-10,226.619995,230.199997,0.015797
7,BRK-B,2020-01-13,228.449997,228.630005,0.000788
8,BRK-B,2020-01-14,227.169998,229.539993,0.010433
9,BRK-B,2020-01-15,228.350006,229.380005,0.004511


In [9]:
print("Missing future closes:")
print(df["future_close_5d"].isna().sum())

print("\nMissing forward returns:")
print(df["forward_return_5d"].isna().sum())

print("\nForward-return summary:")
display(df["forward_return_5d"].describe())

Missing future closes:
25

Missing forward returns:
25

Forward-return summary:


count    7510.000000
mean        0.006259
std         0.048637
min        -0.336987
25%        -0.018273
50%         0.005663
75%         0.030307
max         0.330894
Name: forward_return_5d, dtype: float64

In [10]:
forward_return_by_ticker = (
    df.groupby("ticker")["forward_return_5d"]
    .describe().T
)

display(forward_return_by_ticker)

ticker,BRK-B,CVX,GE,MSFT,NVDA
count,1502.000000,1502.000000,1502.000000,1502.000000,1502.000000
mean,0.003024,0.002719,0.006992,0.004556,0.014004
std,0.027530,0.045813,0.051966,0.036061,0.069927
min,-0.157877,-0.336987,-0.282026,-0.163649,-0.221962
25%,-0.012537,-0.018350,-0.020413,-0.017044,-0.030113
50%,0.003603,0.002963,0.008171,0.005973,0.013240
75%,0.018903,0.025189,0.035584,0.026174,0.055759
max,0.129834,0.330894,0.291326,0.178335,0.302312


In [11]:
largest_positive_returns = (
    df[
        [
            "ticker",
            "date",
            "Close",
            "future_close_5d",
            "forward_return_5d"
        ]
    ]
    .sort_values(
        "forward_return_5d",
        ascending=False
    )
    .head(10)
)

largest_negative_returns = (
    df[
        [
            "ticker",
            "date",
            "Close",
            "future_close_5d",
            "forward_return_5d"
        ]
    ]
    .sort_values(
        "forward_return_5d",
        ascending=True
    )
    .head(10)
)

print("Largest positive five-day returns:")
display(largest_positive_returns)

print("Largest negative five-day returns:")
display(largest_negative_returns)

Largest positive five-day returns:


,ticker,date,Close,future_close_5d,forward_return_5d
1560,CVX,2020-03-19,43.598209,58.024578,0.330894
1562,CVX,2020-03-23,41.190006,54.659180,0.327001
6882,NVDA,2023-05-24,30.475471,39.688568,0.302312
3069,GE,2020-03-23,29.568130,38.182091,0.291326
6880,NVDA,2023-05-22,31.112165,40.028873,0.286599
1559,CVX,2020-03-18,41.820553,52.623249,0.258311
1569,CVX,2020-04-01,52.083874,65.317551,0.254084
6581,NVDA,2022-03-14,21.266453,26.654350,0.253352
3067,GE,2020-03-19,31.358671,39.295139,0.253087
3117,GE,2020-06-01,32.713680,40.940487,0.251479


Largest negative five-day returns:


,ticker,date,Close,future_close_5d,forward_return_5d
1554,CVX,2020-03-11,63.076481,41.820553,-0.336987
1556,CVX,2020-03-13,63.372768,45.117577,-0.288061
3057,GE,2020-03-05,48.731770,34.988155,-0.282026
3054,GE,2020-03-02,54.194752,39.730663,-0.266891
3056,GE,2020-03-04,52.937782,39.730663,-0.249484
1555,CVX,2020-03-12,57.933437,43.598209,-0.247443
1557,CVX,2020-03-16,52.949902,41.190006,-0.222095
6695,NVDA,2022-08-25,17.863405,13.898415,-0.221962
1550,CVX,2020-03-05,73.833595,57.933437,-0.215351
6071,NVDA,2020-03-05,6.797644,5.380360,-0.208496


In [12]:
candidate_thresholds = [
    0.005,   # ±0.5%
    0.010,   # ±1.0%
    0.015,   # ±1.5%
    0.020    # ±2.0%
]

threshold_results = []

valid_returns = df["forward_return_5d"].dropna()

for threshold in candidate_thresholds:

    movement = np.select(
        [valid_returns > threshold, valid_returns < -threshold],["Up", "Down"], default="Neutral")

    counts = pd.Series(movement).value_counts()

    threshold_results.append(
        {
            "threshold": threshold,
            "down": counts.get("Down", 0),
            "neutral": counts.get("Neutral", 0),
            "up": counts.get("Up", 0)
        }
    )

threshold_comparison = pd.DataFrame(
    threshold_results
)

threshold_comparison["threshold_percentage"] = (
    threshold_comparison["threshold"] * 100
)

display(
    threshold_comparison[
        [
            "threshold_percentage",
            "down",
            "neutral",
            "up"
        ]
    ]
)

,threshold_percentage,down,neutral,up
0,0.5,2810,894,3806
1,1.0,2451,1672,3387
2,1.5,2078,2458,2974
3,2.0,1777,3156,2577


In [13]:
class_columns = [
    "down",
    "neutral",
    "up"
]

for column in class_columns:
    threshold_comparison[
        f"{column}_percentage"
    ] = (
        threshold_comparison[column]
        / threshold_comparison[class_columns].sum(axis=1)
        * 100
    )

display(
    threshold_comparison[
        [
            "threshold_percentage",
            "down_percentage",
            "neutral_percentage",
            "up_percentage"
        ]
    ].round(2)
)

,threshold_percentage,down_percentage,neutral_percentage,up_percentage
0,0.5,37.42,11.90,50.68
1,1.0,32.64,22.26,45.10
2,1.5,27.67,32.73,39.60
3,2.0,23.66,42.02,34.31


## Threshold Comparison Result

The ±1.5% threshold provides the most balanced three-class distribution,
with approximately 27.67% Down, 32.73% Neutral, and 39.60% Up observations.

Smaller thresholds produce a very limited Neutral class, while the ±2.0%
threshold makes Neutral the largest class. Therefore, ±1.5% is selected
as the primary threshold, while ±1.0% and ±2.0% will be retained for
robustness analysis.

In [14]:
candidate_threshold = 0.015

def create_movement_label(value, threshold):
    if pd.isna(value):
        return np.nan
    elif value > threshold:
        return "Up"
    elif value < -threshold:
        return "Down"
    else:
        return "Neutral"

df["movement_5d"] = df["forward_return_5d"].apply(
    lambda value: create_movement_label(
        value,
        candidate_threshold
    )
)

class_distribution_by_ticker = pd.crosstab(
    df["ticker"],
    df["movement_5d"],
    normalize="index"
).mul(100).round(2)

display(class_distribution_by_ticker)

movement_5d,Down,Neutral,Up
ticker,,,
BRK-B,21.90,47.74,30.36
CVX,27.96,36.42,35.62
GE,29.16,26.83,44.01
MSFT,26.90,34.15,38.95
NVDA,32.42,18.51,49.07


In [15]:
# Use a common ±1.5% threshold for all five stocks.
#
# A common threshold supports fair cross-stock comparison and
# leave-one-stock-out evaluation.

final_threshold = 0.015

df["movement_5d"] = np.select(
    [
        df["forward_return_5d"] > final_threshold,
        df["forward_return_5d"] < -final_threshold
    ],
    [
        "Up",
        "Down"
    ],
    default="Neutral"
)

# Keep missing labels for rows where the future return is unavailable.
df.loc[
    df["forward_return_5d"].isna(),
    "movement_5d"
] = np.nan

In [16]:
target_mapping = {
    "Down": 0,
    "Neutral": 1,
    "Up": 2
}

df["target_5d"] = (
    df["movement_5d"]
    .map(target_mapping)
)

In [17]:
display(
    df[
        [
            "ticker",
            "date",
            "forward_return_5d",
            "movement_5d",
            "target_5d"
        ]
    ].head(15)
)

,ticker,date,forward_return_5d,movement_5d,target_5d
0,BRK-B,2020-01-02,0.001138,Neutral,1.0
1,BRK-B,2020-01-03,0.001945,Neutral,1.0
2,BRK-B,2020-01-06,0.006432,Neutral,1.0
3,BRK-B,2020-01-07,0.005533,Neutral,1.0
4,BRK-B,2020-01-08,0.010443,Neutral,1.0
5,BRK-B,2020-01-09,0.004723,Neutral,1.0
6,BRK-B,2020-01-10,0.015797,Up,2.0
7,BRK-B,2020-01-13,0.000788,Neutral,1.0
8,BRK-B,2020-01-14,0.010433,Neutral,1.0
9,BRK-B,2020-01-15,0.004511,Neutral,1.0


In [18]:
model_df = (
    df.dropna(
        subset=[
            "forward_return_5d",
            "movement_5d",
            "target_5d"
        ]
    )
    .copy()
    .reset_index(drop=True)
)

print("Rows before target removal:", len(df))
print("Rows after target removal:", len(model_df))
print("Rows removed:", len(df) - len(model_df))

Rows before target removal: 7535
Rows after target removal: 7510
Rows removed: 25


In [19]:
print("Target counts:")
print(
    model_df["movement_5d"]
    .value_counts()
)

print("\nTarget percentages:")
print(
    model_df["movement_5d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissing target values:")
print(
    model_df[
        [
            "forward_return_5d",
            "movement_5d",
            "target_5d"
        ]
    ].isna().sum()
)

Target counts:
movement_5d
Up         2974
Neutral    2458
Down       2078
Name: count, dtype: int64

Target percentages:
movement_5d
Up         39.60
Neutral    32.73
Down       27.67
Name: proportion, dtype: float64

Missing target values:
forward_return_5d    0
movement_5d          0
target_5d            0
dtype: int64


In [20]:
technical_features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "SMA_50",
    "SMA_200",
    "EMA_50",
    "EMA_200",
    "RSI",
    "MACD",
    "MACD_Signal",
    "MACD_Hist",
    "BB_Upper",
    "BB_Middle",
    "BB_Lower",
    "ATR",
    "ADX"
]

gdelt_features = [
    "gdelt_tone_lag_1",
    "gdelt_tone_mean_3",
    "gdelt_tone_mean_5",
    "gdelt_tone_std_5",
    "gdelt_tone_change_1d",
    "gdelt_abnormal_tone_20",
    "gdelt_tone_missing"
]

market_features = [
    "market_return_lag_1",
    "vix_close_lag_1",
    "vix_change_lag_1"
]

all_model_features = (
    technical_features
    + gdelt_features
    + market_features
)

print("Number of technical features:", len(technical_features))
print("Number of GDELT features:", len(gdelt_features))
print("Number of market features:", len(market_features))
print("Total model features:", len(all_model_features))

Number of technical features: 18
Number of GDELT features: 7
Number of market features: 3
Total model features: 28


In [21]:
missing_columns = [
    column
    for column in all_model_features
    if column not in model_df.columns
]

print("Undefined dataset columns:", missing_columns)

Undefined dataset columns: []


In [22]:
all_model_features = (
    technical_features
    + gdelt_features
    + market_features
)

missing_feature_summary = (
    model_df[all_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_feature_summary[
        missing_feature_summary > 0
    ]
)

gdelt_abnormal_tone_20    135
gdelt_tone_change_1d       85
gdelt_tone_lag_1           70
gdelt_tone_std_5           55
gdelt_tone_mean_3          50
gdelt_tone_mean_5          45
market_return_lag_1        10
vix_change_lag_1           10
vix_close_lag_1             5
dtype: int64

In [23]:
rows_with_missing_features = (
    model_df[all_model_features]
    .isna()
    .any(axis=1)
)

print(
    "Rows with at least one missing model feature:",
    rows_with_missing_features.sum()
)

print(
    "Percentage of affected rows:",
    round(
        rows_with_missing_features.mean() * 100,
        2
    )
)

Rows with at least one missing model feature: 140
Percentage of affected rows: 1.86


In [24]:
missing_feature_summary = (
    model_df[all_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_feature_summary[
        missing_feature_summary > 0
    ]
)

gdelt_abnormal_tone_20    135
gdelt_tone_change_1d       85
gdelt_tone_lag_1           70
gdelt_tone_std_5           55
gdelt_tone_mean_3          50
gdelt_tone_mean_5          45
market_return_lag_1        10
vix_change_lag_1           10
vix_close_lag_1             5
dtype: int64

In [25]:
missing_columns = [
    column
    for column in all_model_features
    if column not in model_df.columns
]

print("Undefined dataset columns:", missing_columns)

Undefined dataset columns: []


In [26]:
complete_model_df = (
    model_df
    .dropna(subset=all_model_features)
    .copy()
    .reset_index(drop=True)
)

print("Rows before feature removal:", len(model_df))
print("Rows after feature removal:", len(complete_model_df))
print("Rows removed:", len(model_df) - len(complete_model_df))

print(
    "Remaining percentage:",
    round(
        len(complete_model_df) / len(model_df) * 100,
        2
    )
)

Rows before feature removal: 7510
Rows after feature removal: 7370
Rows removed: 140
Remaining percentage: 98.14


In [27]:
print(
    "Remaining missing feature values:",
    complete_model_df[all_model_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Remaining missing target values:",
    complete_model_df[
        [
            "forward_return_5d",
            "movement_5d",
            "target_5d"
        ]
    ]
    .isna()
    .sum()
    .sum()
)

Remaining missing feature values: 0
Remaining missing target values: 0


In [28]:
print("Final target counts:")
print(
    complete_model_df["movement_5d"]
    .value_counts()
)

print("\nFinal target percentages:")
print(
    complete_model_df["movement_5d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nRows per ticker:")
print(
    complete_model_df["ticker"]
    .value_counts()
    .sort_index()
)

Final target counts:
movement_5d
Up         2911
Neutral    2414
Down       2045
Name: count, dtype: int64

Final target percentages:
movement_5d
Up         39.50
Neutral    32.75
Down       27.75
Name: proportion, dtype: float64

Rows per ticker:
ticker
BRK-B    1474
CVX      1474
GE       1474
MSFT     1474
NVDA     1474
Name: count, dtype: int64


In [29]:
# Convert the numeric target to integer form.
complete_model_df["target_5d"] = (
    complete_model_df["target_5d"]
    .astype(int)
)

full_output_file = (
    "/kaggle/working/"
    "model_ready_with_missing_features_2020_2025.csv"
)

complete_output_file = (
    "/kaggle/working/"
    "model_ready_complete_case_2020_2025.csv"
)

model_df.to_csv(
    full_output_file,
    index=False
)

complete_model_df.to_csv(
    complete_output_file,
    index=False
)

print("Saved full labelled dataset:")
print(full_output_file)

print("\nSaved complete-case modelling dataset:")
print(complete_output_file)

print("\nFinal complete-case shape:")
print(complete_model_df.shape)

Saved full labelled dataset:
/kaggle/working/model_ready_with_missing_features_2020_2025.csv

Saved complete-case modelling dataset:
/kaggle/working/model_ready_complete_case_2020_2025.csv

Final complete-case shape:
(7370, 42)


In [30]:
saved_df = pd.read_csv(
    complete_output_file,
    parse_dates=["date"]
)

print("Reloaded shape:", saved_df.shape)

print(
    "Duplicate ticker-date rows:",
    saved_df.duplicated(
        subset=["ticker", "date"]
    ).sum()
)

print(
    "Missing selected feature values:",
    saved_df[all_model_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Missing target values:",
    saved_df["target_5d"]
    .isna()
    .sum()
)

print(
    "Target values:",
    sorted(saved_df["target_5d"].unique())
)

Reloaded shape: (7370, 42)
Duplicate ticker-date rows: 0
Missing selected feature values: 0
Missing target values: 0
Target values: [np.int64(0), np.int64(1), np.int64(2)]
